# TF-IDF Retrieval

This notebook implements the retrieval component of the retrieval-augmented generation (RAG) pipeline.

## Purpose

TF-IDF is used as a sparse retrieval baseline to retrieve the course-material chunks that are most relevant to a student question. The retrieved chunks can later be passed to an LLM or a sequence-to-sequence model to generate an answer.

## Pipeline

1. Load the preprocessed course-material chunks created in Notebook 03.
2. Build a TF-IDF index from the course-material chunks.
3. Retrieve the top-$k$ chunks for each question using cosine similarity.
4. Evaluate retrieval performance against the annotated source pages.
5. Use the retrieved chunks as context for the LLM and sequence-to-sequence experiments.

- **Train:** retrieved contexts are used as input to the LLM and sequence-to-sequence experiments.
- **Validation:** retrieved contexts are used to tune and evaluate the retrieval and generation pipeline.
- **Test:** retrieved contexts are used for the final, held-out evaluation.

## Data flow

- **Input documents:** `data/processed/chunks_preprocessed.jsonl`
- **Questions:** `data/splits/train.jsonl`, `data/splits/validation.jsonl`, and `data/splits/test.jsonl`
- **Retriever:** TF-IDF with cosine similarity
- **Output:** ranked chunks, retrieved page references, and retrieval metrics

The TF-IDF retriever is fitted on the course-material chunks. The same fitted retriever is then used to retrieve relevant chunks for the train, validation, and test questions.

## RAG Retrieval Pipeline

```text
Notebook 03
Preprocessing and splitting
        |
        v
Preprocessed course chunks
        |
        v
Notebook 04
TF-IDF retriever
        |
        +------------------+
        |                  |
        v                  v
Train questions      Validation questions
retrieved contexts   tune and evaluate
        |
        v
Answer generator
        |
        +------------------+
        |                  |
        v                  v
      LLM              Seq2Seq
        |                  |
        +--------+---------+
                 v
              Answers

Test questions
        |
        v
Final retrieval evaluation
        |
        v
Final LLM and Seq2Seq evaluation
```

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
TOP_K_VALUES = (1, 3, 5, 10)

In [3]:
PROJECT_ROOT = Path.cwd()

PREPROCESSING_NOTEBOOK_PATH = (
    PROJECT_ROOT
    / "03_proccessingSplitting.ipynb"
)

CHUNKS_PREPROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)

TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "train.jsonl"
)

VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "validation.jsonl"
)

TEST_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "test.jsonl"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "retrieval"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

assert PREPROCESSING_NOTEBOOK_PATH.exists()
assert CHUNKS_PREPROCESSED_PATH.exists()
assert TRAIN_PATH.exists()
assert VALIDATION_PATH.exists()
assert TEST_PATH.exists()

print("Notebook 03:", PREPROCESSING_NOTEBOOK_PATH)
print("Chunkovi:", CHUNKS_PREPROCESSED_PATH)
print("Trening skup:", TRAIN_PATH)
print("Validacioni skup:", VALIDATION_PATH)
print("Test skup:", TEST_PATH)

Notebook 03: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/03_proccessingSplitting.ipynb
Chunkovi: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/processed/chunks_preprocessed.jsonl
Trening skup: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/train.jsonl
Validacioni skup: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/validation.jsonl
Test skup: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/splits/test.jsonl


In [4]:
with PREPROCESSING_NOTEBOOK_PATH.open("r", encoding="utf-8") as file:
    notebook_03 = json.load(file)

load_jsonl_source = next(
    cell["source"]
    for cell in notebook_03["cells"]
    if cell.get("cell_type") == "code"
    and any(
        line.startswith("def load_jsonl")
        for line in cell.get("source", [])
    )
)

exec("".join(load_jsonl_source), globals())
print("Funkcija load_jsonl je učitana iz notebooka 03.")

Funkcija load_jsonl je učitana iz notebooka 03.


In [5]:
chunks = load_jsonl(CHUNKS_PREPROCESSED_PATH)

train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

print(f"Broj chunkova: {len(chunks)}")
print(f"Broj pitanja u trening skupu: {len(train_data)}")
print(f"Broj pitanja u validacionom skupu: {len(validation_data)}")
print(f"Broj pitanja u test skupu: {len(test_data)}")

Broj chunkova: 356
Broj pitanja u trening skupu: 100
Broj pitanja u validacionom skupu: 21
Broj pitanja u test skupu: 22


In [6]:
assert chunks

assert all(
    isinstance(chunk.get("lexical_text"), str)
    and chunk["lexical_text"].strip()
    for chunk in chunks
)

for split_name, data in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data
}.items():

    assert data, f"{split_name} split is empty"

    assert all(
        isinstance(example.get("lexical_question"), str)
        and example["lexical_question"].strip()
        for example in data
    ), f"Missing lexical_question in {split_name}"

In [7]:
chunks_df = pd.DataFrame(chunks)

train_df = pd.DataFrame(train_data)
validation_df = pd.DataFrame(validation_data)
test_df = pd.DataFrame(test_data)

chunks_df[
    ["chunk_id", "pdf_page_start", "pdf_page_end", "lexical_text"]
].head(3)

,chunk_id,pdf_page_start,pdf_page_end,lexical_text
0,chunk_0001,17,17,pregled 1 1 upravljanje kvalitetom softvera 4 ...
1,chunk_0002,17,18,nu ulogu u razvoju veštačke inteligencije obra...
2,chunk_0003,17,18,koriste kako bi se na vreme zadovoljili korisn...


In [8]:
chunk_ids = chunks_df["chunk_id"].tolist()

chunk_texts = chunks_df["lexical_text"].tolist()

print(f"Broj chunkova: {len(chunk_texts)}")
print(f"Broj ID-ova chunkova: {len(chunk_ids)}")

Broj chunkova: 356
Broj ID-ova chunkova: 356


## TF-IDF Vectorization

The document chunks are converted into TF-IDF vectors using unigrams and bigrams. Serbian stop words are removed so that common function words contribute less to retrieval scores.

In [9]:
serbian_stop_words = {
    "i", "u", "na", "je", "su", "se", "za", "od", "do",
    "sa", "kao", "koji", "koja", "koje", "što", "da",
    "ili", "a", "ali", "po", "iz", "uz", "o", "pri",
    "bi", "biti", "ima", "imaju"
}

vectorizer = TfidfVectorizer(
    lowercase=False,
    stop_words=list(serbian_stop_words),
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

In [10]:
tfidf_matrix = vectorizer.fit_transform(chunk_texts)

print("Oblik matrice TF-IDF :", tfidf_matrix.shape)
print("Velicina rečnika:", len(vectorizer.vocabulary_))
print(f"Broj nenultih vrednosti: {tfidf_matrix.nnz}")

Oblik matrice TF-IDF : (356, 34060)
Velicina rečnika: 34060
Broj nenultih vrednosti: 70365


## Inspect TF-IDF Terms

This section shows which terms receive the highest TF-IDF weights in one document chunk. These weights provide an interpretable view of the words and bigrams that characterize that chunk.

In [11]:
chunk_index = 0

feature_names = np.array(
    vectorizer.get_feature_names_out()
)

vector = tfidf_matrix[chunk_index]

scores = vector.toarray().flatten()

top_indices = scores.argsort()[::-1][:15]

tfidf_terms_df = pd.DataFrame({
    "term": feature_names[top_indices],
    "tfidf": scores[top_indices]
})

tfidf_terms_df

,term,tfidf
0,industrija,0.139983
1,važni,0.108605
2,standardi,0.103554
3,razvija,0.094377
4,kvaliteta softvera,0.094021
5,atributi kvaliteta,0.086761
6,atributi,0.085708
7,kvalitet softvera,0.082815
8,oni definišu,0.082676
9,zabave,0.082676


In [12]:
example = validation_data[0]

query = example["lexical_question"]

query_vector = vectorizer.transform([query])

query_scores = query_vector.toarray().flatten()

nonzero = np.where(query_scores > 0)[0]

query_terms_df = pd.DataFrame({
    "term": feature_names[nonzero],
    "tfidf": query_scores[nonzero]
}).sort_values(
    "tfidf",
    ascending=False
)

query_terms_df

,term,tfidf
4,šta,0.653766
3,služi,0.543619
0,cachegrind,0.433502
2,koristi,0.215082
1,kako,0.207085


In [13]:
import altair as alt

alt.Chart(query_terms_df).mark_bar().encode(
    x=alt.X(
        "tfidf:Q",
        title="TF-IDF težina"
    ),
    y=alt.Y(
        "term:N",
        sort="-x",
        title="Termin"
    ),
    tooltip=["term", "tfidf"]
).properties(
    title="TF-IDF reprezentacija pitanja",
    width=500,
    height=300
)

alt.Chart(...)

## Single Query Retrieval

A single lexical question is transformed into a TF-IDF vector and compared with every document chunk. The highest-scoring chunks are returned as the query's ranked retrieval results.

In [14]:
def retrieve_chunks(
    lexical_question: str,
    top_k: int = 5
) -> pd.DataFrame:

    question_vector = vectorizer.transform(
        [lexical_question]
    )

    if question_vector.nnz == 0:
        return pd.DataFrame(columns=[
            "rank",
            "chunk_id",
            "score",
            "pdf_page_start",
            "pdf_page_end",
            "processed_text",    # bolja čitljivost
        ])
    
    similarities = cosine_similarity(
        question_vector,
        tfidf_matrix
    ).flatten()

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        chunk = chunks[index]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "score": float(similarities[index]),
            "pdf_page_start": chunk["pdf_page_start"],
            "pdf_page_end": chunk["pdf_page_end"],
            "processed_text": chunk["processed_text"]   # bolja čitljivost
        })

    return pd.DataFrame(results)

In [15]:
def get_relevant_chunk_ids(gold_pages):
    if isinstance(gold_pages, int):
        gold_pages = [gold_pages]

    gold_pages = set(gold_pages)
    relevant_ids = set()

    for chunk in chunks:
        chunk_pages = set(
            range(
                int(chunk["pdf_page_start"]),
                int(chunk["pdf_page_end"]) + 1
            )
        )

        if chunk_pages & gold_pages:
            relevant_ids.add(chunk["chunk_id"])

    return relevant_ids

In [16]:
example = validation_data[1]

print("Original question:")
print(example["question"])

print("\nProcessed question:")
print(example["lexical_question"])

print("\nGold source pages:")
print(example["source_pages"])

Original question:
Šta je instrumentaciono profajliranje?

Processed question:
šta je instrumentaciono profajliranje

Gold source pages:
[188, 189]


In [17]:
def overlaps_gold_pages(
    row,
    source_pages
):
    return any(
        row["pdf_page_start"] <= page <= row["pdf_page_end"]
        for page in source_pages
    )

In [18]:
retrieved = retrieve_chunks(
    example["lexical_question"],
    top_k=10
)

retrieved["relevant"] = retrieved.apply(
    lambda row: overlaps_gold_pages(
        row,
        example["source_pages"]
    ),
    axis=1
)

retrieved[
    [
        "rank",
        "score",
        "pdf_page_start",
        "pdf_page_end",
        "relevant",
        "processed_text"
    ]
]

,rank,score,pdf_page_start,pdf_page_end,relevant,processed_text
0,1,0.108668,185,185,False,[Profajleri] ogućava programu da se izvršava g...
1,2,0.108119,188,188,True,[Profajliranje i dinamičko detektovanje grešak...
2,3,0.053569,174,175,False,[Profajliranje i dinamičko detektovanje grešak...
3,4,0.047209,172,173,False,"ovanja, tačaka prekida i tačaka posmatranja. P..."
4,5,0.034037,188,189,True,[Profajliranje i dinamičko detektovanje grešak...
5,6,0.029150,189,189,True,[Profajleri] i prati ponašanje programa u real...
6,7,0.027817,58,61,False,Primeri. 9. Tehnike verifikacije softvera. Osn...
7,8,0.027412,204,204,False,[Profajliranje i dinamičko detektovanje grešak...
8,9,0.027118,181,181,False,"[Profajleri] čite informacije, uključujući fre..."
9,10,0.026372,184,184,False,[Profajliranje i dinamičko detektovanje grešak...


## Visualize Similarities

Cosine similarity measures how closely the question matches each chunk in the TF-IDF space. The top 20 scores make the retrieval ranking visible and show how sharply the best matches stand out.

In [19]:
example = validation_data[0]

question_vector = vectorizer.transform(
    [example["lexical_question"]]
)

similarities = cosine_similarity(
    question_vector,
    tfidf_matrix
).flatten()

similarity_df = pd.DataFrame({
    "chunk_index": range(len(similarities)),
    "similarity": similarities
}).sort_values(
    "similarity",
    ascending=False
)

top_similarity_df = similarity_df.head(20)

top_similarity_df

,chunk_index,similarity
296,296,0.093111
293,293,0.083113
286,286,0.056625
338,338,0.055103
339,339,0.049709
135,135,0.049207
88,88,0.048117
87,87,0.047405
335,335,0.046490
89,89,0.043297


In [20]:
alt.Chart(top_similarity_df).mark_bar().encode(
    x=alt.X(
        "similarity:Q",
        title="Cosine similarity"
    ),
    y=alt.Y(
        "chunk_index:O",
        sort="-x",
        title="Chunk"
    ),
    tooltip=["chunk_index", "similarity"]
).properties(
    title="TF-IDF similarity pitanja sa top chunkovima",
    width=500,
    height=400
)

alt.Chart(...)

## Retrieval Evaluation

The retriever is evaluated by comparing the retrieved chunk IDs with the chunks whose page ranges overlap the annotated source pages. This produces per-query relevance values that are aggregated across the validation set.

In [21]:
def calculate_metrics_at_k(
    relevances,
    n_relevant,
    k
):
    rel = np.array(relevances[:k], dtype=int)

    n_retrieved_relevant = int(rel.sum())

    # Hit@k - da li smo našli makar jedan relevantan chunk
    hit = int(n_retrieved_relevant > 0)

    # Precision@k - koliko od top-k je relevantno
    precision = n_retrieved_relevant / k

    # Recall@k - koliko relevantnih chunkova smo pokrili
    recall = (
        n_retrieved_relevant / n_relevant
        if n_relevant > 0
        else 0.0
    )

    # F1@k - harmonijska sredina precisiona i recall-a
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    # MRR@k - koliko visoko se pojavio prvi relevantan chunk
    reciprocal_rank = 0.0

    for rank, relevant in enumerate(rel, start=1):
        if relevant:
            reciprocal_rank = 1.0 / rank
            break

    # nDCG@k - kvalitet celog redosleda relevantnih chunkova
    dcg = sum(
        relevant / np.log2(rank + 1)
        for rank, relevant in enumerate(rel, start=1)
    )

    ideal_relevant = min(n_relevant, k)

    idcg = sum(
        1.0 / np.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        "Hit": hit,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "MRR": reciprocal_rank,
        "nDCG": ndcg,
    }

In [22]:
def evaluate_retriever(
    data,
    top_k_values=TOP_K_VALUES
):
    max_k = max(top_k_values)

    per_query_results = []
    skipped_questions = []

    for example in data:

        relevant_ids = get_relevant_chunk_ids(
            example["source_pages"]
        )

        # Pitanje nije evaluabilno ako corpus nema nijedan
        # chunk koji pokriva njegove gold stranice
        if not relevant_ids:
            skipped_questions.append(example["id"])
            continue

        # Retrieval radimo SAMO JEDNOM
        retrieved = retrieve_chunks(
            example["lexical_question"],
            top_k=max_k
        )

        retrieved_ids = retrieved["chunk_id"].tolist()

        relevances = [
            1 if chunk_id in relevant_ids else 0
            for chunk_id in retrieved_ids
        ]

        for k in top_k_values:

            metrics = calculate_metrics_at_k(
                relevances=relevances,
                n_relevant=len(relevant_ids),
                k=k
            )

            per_query_results.append({
                "question_id": example["id"],
                "k": k,
                **metrics
            })

    per_query_df = pd.DataFrame(per_query_results)

    metrics_df = (
        per_query_df
        .groupby("k")[
            [
                "Hit",
                "Precision",
                "Recall",
                "F1",
                "MRR",
                "nDCG"
            ]
        ]
        .mean()
        .reset_index()
    )

    print(f"Ukupno pitanja: {len(data)}")
    print(f"Evaluabilno: {len(data) - len(skipped_questions)}")
    print(f"Preskočeno: {len(skipped_questions)}")

    if skipped_questions:
        print("Preskočeni question IDs:", skipped_questions)

    return metrics_df, per_query_df

## Hit@k / Precision@k / Recall@k / F1@k / MRR@k / nDCG@k

These metrics describe complementary aspects of retrieval quality. 

**Hit@k** measures whether at least one relevant chunk is retrieved, 

**Precision@k** measures the proportion of relevant chunks among the top-$k$ results, and 

**Recall@k** measures how much of the relevant material is retrieved. 

**F1@k** is the harmonic mean of Precision@k and Recall@k. 

**MRR@k** measures how high the first relevant result appears, while 

**nDCG@k** evaluates the quality of the complete ranking, giving greater importance to relevant chunks appearing near the top. 

The metrics are reported for several values of $k$ to show how retrieval quality changes as more chunks are considered.

In [23]:
metrics_df, per_query_metrics_df = evaluate_retriever(
    validation_data
)

metrics_df

Ukupno pitanja: 21
Evaluabilno: 21
Preskočeno: 0


,k,Hit,Precision,Recall,F1,MRR,nDCG
0,1,0.285714,0.285714,0.059921,0.097884,0.285714,0.285714
1,3,0.571429,0.317460,0.211508,0.248196,0.396825,0.308061
2,5,0.809524,0.333333,0.354009,0.333272,0.453968,0.347742
3,10,0.857143,0.223810,0.452896,0.291901,0.459259,0.385711


In [24]:
import altair as alt

alt.Chart(tfidf_terms_df).mark_bar().encode(
    x=alt.X(
        "tfidf:Q",
        title="TF-IDF težina"
    ),
    y=alt.Y(
        "term:N",
        sort="-x",
        title="Termin"
    ),
    tooltip=["term", "tfidf"]
).properties(
    title="Najvažniji TF-IDF termini u chunku",
    width=500,
    height=350
)

alt.Chart(...)

In [25]:
metrics_plot_df = metrics_df.melt(
    id_vars="k",
    value_vars=[
        "Hit",
        "Precision",
        "Recall",
        "F1",
        "MRR",
        "nDCG"
    ],
    var_name="metric",
    value_name="score"
)

chart = (
    alt.Chart(metrics_plot_df)
    .mark_line(point=True)
    .encode(
        x=alt.X(
            "k:O",
            title="Top-k"
        ),
        y=alt.Y(
            "score:Q",
            title="Score",
            scale=alt.Scale(domain=[0, 1])
        ),
        color=alt.Color(
            "metric:N",
            title="Metrika"
        ),
        tooltip=[
            "k",
            "metric",
            alt.Tooltip(
                "score:Q",
                format=".3f"
            )
        ]
    )
    .properties(
        title="TF-IDF Retrieval Performance",
        width=600,
        height=350
    )
)

chart

alt.Chart(...)

## Saving TF-IDF Artifacts

The fitted vectorizer, TF-IDF matrix, chunk metadata, validation metrics, and top-k retrieval results are saved for reuse. These artifacts are providing the shared retrieval input for the Qwen and mT5 experiments.

In [26]:
import json

import joblib
from scipy import sparse

ARTIFACTS_DIR = RESULTS_DIR / "tfidf"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    vectorizer,
    ARTIFACTS_DIR / "tfidf_vectorizer.joblib"
)

sparse.save_npz(
    ARTIFACTS_DIR / "tfidf_matrix.npz",
    tfidf_matrix
)

chunks_export = chunks_df[
    [
        "chunk_id",
        "text",
        "processed_text",
        "lexical_text",
        "pdf_page_start",
        "pdf_page_end",
        "printed_page_start",
        "printed_page_end",
        "section_ref",
        "heading",
    ]
]

chunks_export.to_json(
    ARTIFACTS_DIR / "tfidf_chunks.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

metadata = {
    "vectorizer_input": "lexical_text",
    "retrieval_question_input": "lexical_question",
    "generator_context_input": "processed_text",
    "top_k": max(TOP_K_VALUES),
    "n_chunks": len(chunks),
    "n_features": len(vectorizer.get_feature_names_out()),
}

with (ARTIFACTS_DIR / "tfidf_metadata.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

metrics_df.to_csv(
    ARTIFACTS_DIR / "validation_metrics.csv",
    index=False
)

per_query_metrics_df.to_csv(
    ARTIFACTS_DIR / "validation_per_query_metrics.csv",
    index=False
)

print("TF-IDF artefakti su sačuvani u:", ARTIFACTS_DIR)

TF-IDF artefakti su sačuvani u: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/retrieval/tfidf


In [27]:
def export_retrieval_results(data, split_name, top_k=10):
    export_path = RESULTS_DIR / f"tfidf_{split_name}_top{top_k}.jsonl"

    with export_path.open("w", encoding="utf-8") as file:
        for example in data:
            retrieved = retrieve_chunks(
                example["lexical_question"],
                top_k=top_k
            )

            record = {
                "question_id": example["id"],
                "question": example["question"],
                "processed_question": example["processed_question"],
                "lexical_question": example["lexical_question"],
                "answer": example["answer"],
                "source_pages": example["source_pages"],
                "retrieved_chunks": retrieved.to_dict(orient="records"),
            }

            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    print(f"Sačuvani retrieval rezultati za {split_name}:", export_path)


for split_name, data in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
}.items():
    export_retrieval_results(
        data,
        split_name,
        top_k=max(TOP_K_VALUES)
    )

Sačuvani retrieval rezultati za train: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/retrieval/tfidf_train_top10.jsonl
Sačuvani retrieval rezultati za validation: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/retrieval/tfidf_validation_top10.jsonl
Sačuvani retrieval rezultati za test: /home/julijana/Desktop/Student-Question-Answering-from-Course-Materials/data/retrieval/tfidf_test_top10.jsonl
